### Installing Dependencies
First, we install the necessary libraries for our environment, including `transformers` and `sentence-transformers`.

In [ ]:
#%%capture
#!pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

Downloading and Running An LLM
The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately and keep them as such so that we can explore them separately.

### Loading the Model and Tokenizer
We load the Phi-3-mini-4k-instruct model and its corresponding tokenizer. The model is loaded onto the GPU (`device_map="cuda"`) for faster generation.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

### Generating Text
Here we define a prompt and tokenize it. The tokenizer converts the text into token IDs (`input_ids`) that the model can understand. Then, we use `model.generate` to produce the response.

In [9]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"


# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=20
)

# Print the output
print(tokenizer.decode(generation_output[0]))

Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Heartfelt Apologies for the Gardening Mishap


Dear


### Inspecting Token IDs
Let's print the raw token IDs generated by the tokenizer for our prompt.

In [10]:
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


### Decoding Individual Tokens
By decoding each token ID one by one, we can see exactly how the tokenizer split our input prompt into smaller pieces (tokens).

In [11]:
for id in input_ids[0]:
   print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


### Model Output Tensor
This is the raw tensor output from the model's generation step. It contains the token IDs of both our input prompt and the generated response.

In [12]:
generation_output

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='cuda:0')

### Understanding Special/Specific Tokens
We can decode specific token IDs to understand how the model groups characters. Notice how 'Subject' is split or merged depending on the tokenizer's vocabulary.

In [13]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


#Comparing Trained LLM Tokenizers

### Visualization Helper Function
We define a helper function `show_tokens` that prints each token in a different color. This helps visually compare how different tokenizers split the same text.

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

### Sample Text for Tokenizer Comparison
We define a challenging text sample containing mixed capitalization, emojis, code snippets, and numbers to test different tokenizers.

In [15]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

### BERT Uncased Tokenizer
The `bert-base-uncased` tokenizer converts everything to lowercase before tokenization. Notice how it handles punctuation and subwords (marked with `##`).

In [16]:
show_tokens(text, "bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 [SEP] 

### BERT Cased Tokenizer
Unlike the uncased version, `bert-base-cased` preserves capitalization. This leads to a different set of tokens for capitalized words.

In [17]:
show_tokens(text, "bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 [SEP] 

### GPT-2 Tokenizer
The GPT-2 tokenizer uses Byte-Pair Encoding (BPE). Notice how it handles whitespace (represented by `Ġ` or special characters) differently than BERT.

In [18]:
show_tokens(text, "gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"        "  Three  tabs :  "              " 
 12 . 0 * 50 = 600 
 

### GPT-4 Tokenizer (tiktoken)
This uses the `tiktoken` equivalent for GPT-4. It is often much more efficient at packing text (fewer tokens for the same text) compared to older models.

In [19]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")

tokenizer_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


 English  and  CAPITAL IZATION 
 � � �  � � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 12 . 0 * 50 = 600 
 

### StarCoder2 Tokenizer
StarCoder2 is heavily optimized for code. It has specific tokens for multiple spaces and code-specific syntax to efficiently tokenize source code.

In [20]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")

config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


 English  and  CAPITAL IZATION 
 � � �   � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

### Phi-3 Tokenizer
Finally, we see how our Phi-3 model tokenizes the text. It uses a SentencePiece-based tokenizer.

In [21]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

 
 English and C AP IT AL IZ ATION 
 � � � �  � � � 
 show _ to kens False None elif == >= else : two tabs :"    " Three tabs : "       " 
 1 2 . 0 * 5 0 = 6 0 0 
 

Contextualized Word Embeddings From a Language Model (Like BERT)

### Contextualized Word Embeddings (DeBERTa)
Instead of generating text, models like BERT and DeBERTa are used to extract contextualized word embeddings. Here we load DeBERTa to inspect the hidden states.

In [22]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/a

model.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

### Output Shape
The output shape is `(batch_size, sequence_length, hidden_size)`. For our 4 tokens ('Hello', 'world', and special tokens), we get a sequence of 4 embeddings.

In [25]:
output.shape

torch.Size([1, 4, 384])

### DeBERTa Tokens
Printing the tokens shows the special `[CLS]` (start) and `[SEP]` (end) tokens automatically added by the tokenizer.

In [26]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hello
 world
[SEP]


### Raw Embedding Vectors
This tensor contains the actual numerical embeddings (the hidden states) for each token in our sentence.

In [27]:
output

tensor([[[-3.4805,  0.0862, -0.1818,  ..., -0.0610, -0.3909,  0.3022],
         [ 0.1885,  0.3201, -0.2313,  ...,  0.3721,  0.2471,  0.8057],
         [ 0.2089,  0.5010, -0.0495,  ...,  1.2197, -0.2277,  0.8574],
         [-3.4277,  0.0635, -0.1426,  ...,  0.0658, -0.4358,  0.3826]]],
       dtype=torch.float16, grad_fn=<NativeLayerNormBackward0>)

#Text Embeddings (For Sentences and Whole Documents)

### Freeing GPU Memory
Since we are moving on to a new model, we manually run the garbage collector and clear the PyTorch CUDA cache to avoid 'Out of Memory' errors.

In [1]:
import torch
import gc
# Delete the old model if you have one loaded (e.g., the Phi-3 model)
# del old_model 
# Run Python garbage collector and empty the CUDA cache
gc.collect()
torch.cuda.empty_cache()

### Sentence Embeddings
Instead of word-level embeddings, `SentenceTransformer` models encode entire sentences or paragraphs into a single fixed-size vector.

In [3]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device="cpu")

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Sentence Vector Shape
Our single sentence is encoded into a 1D vector of size 768. This vector captures the semantic meaning of the entire sentence.

In [4]:
vector.shape

(768,)

Word Embeddings Beyond LLMs

### Static Word Embeddings (GloVe)
Before LLMs, static word embeddings like Word2Vec and GloVe were standard. Here we download a pre-trained GloVe model using the `gensim` library.

In [6]:
!pip install gensim
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 65.5 MB/s eta 0:00:00:00:0100:01
[==================================================] 100.0% 66.0/66.0MB downloaded


### Finding Similar Words
Because GloVe maps semantically similar words to similar vectors, we can find the closest vectors to 'politician' to see related concepts.

In [8]:
model.most_similar([model['politician']], topn=11)

[('politician', 0.9999999403953552),
 ('businessman', 0.7817543148994446),
 ('mp', 0.7724176645278931),
 ('liberal', 0.7486166954040527),
 ('lawmaker', 0.73398756980896),
 ('jurist', 0.7278751134872437),
 ('parliamentarian', 0.7202853560447693),
 ('conservative', 0.7129015922546387),
 ('elected', 0.7100611329078674),
 ('candidate', 0.7029688954353333),
 ('citizen', 0.7005450129508972)]

#Recommending songs by embeddings

### Loading Playlist Data
We can use embedding techniques for more than just words! Here we download a dataset of music playlists, where each playlist is a sequence of song IDs.

In [9]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

### Inspecting Playlists
Each playlist is represented as a list of song IDs. We will treat these IDs as 'words' and the playlists as 'sentences'.

In [10]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

### Training Word2Vec on Playlists
We train a `Word2Vec` model on the playlists. The model will learn that songs appearing frequently together in playlists should have similar vector representations.

In [11]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

### Finding Similar Songs
Now we can ask the model for songs similar to song #2172. It returns the IDs of the most similar songs based on their co-occurrence patterns.

In [12]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('5549', 0.9955399632453918),
 ('2704', 0.9953387379646301),
 ('2688', 0.9951297044754028),
 ('2976', 0.995013415813446),
 ('6644', 0.9949955344200134),
 ('3096', 0.9948954582214355),
 ('3030', 0.9947513341903687),
 ('2069', 0.9942552447319031),
 ('1785', 0.9942408204078674),
 ('3035', 0.9941936731338501)]

### Decoding Song Recommendations
This helper function maps the recommended song IDs back to their actual titles and artists using our metadata DataFrame.

In [13]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
5549,November Rain,Guns N' Roses
2704,Over The Mountain,Ozzy Osbourne
2688,Holy Diver,Dio
2976,I Don't Know,Ozzy Osbourne
6644,Signs,Tesla


### Song Recommendations (Ozzy Osbourne)
We ask for recommendations for an Ozzy Osbourne song. The model correctly suggests other heavy metal and hard rock songs!

In [14]:
print_recommendations(2172)

,title,artist
id,,
5549,November Rain,Guns N' Roses
2704,Over The Mountain,Ozzy Osbourne
2688,Holy Diver,Dio
2976,I Don't Know,Ozzy Osbourne
6644,Signs,Tesla


### Song Recommendations (Hip-Hop/R&B)
We ask for recommendations for a different song, and the model seamlessly suggests related hip-hop and R&B tracks. The embeddings have successfully learned musical genres!

In [15]:
print_recommendations(842)

,title,artist
id,,
413,If I Ruled The World (Imagine That) (w\/ Laury...,Nas
886,Heartless,Kanye West
6741,Love In This Club (w\/ Young Jeezy),Usher
1560,In Da Club,50 Cent
11331,Take You There,Sean Kingston
